In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# Overview
This notebook's purpose is to dig deeper and document what is inside the `_voxelated.npy` and `_base_voxels.npy` files. These files were 2 outputs from the _voxeling and labeling_ section of the `Mg22_Voxel_pipeline.ipynb` Jupyter notebook, which is responsible for creating the data that goes into the machine learning algorithm. It does so by taking an `.h5` file and creates 12 files, which includes test, train, and validation data, and shuffled and unshuffled voxels. 

The `_voxelated.npy` and `_base_voxels.npy` files are outputted when the `Mg22_Voxel_pipeline.ipynb` notebook finishes breaking the data into voxels and assigning each point to the appropriate `voxel_id` (the `voxel_id` is the _kth_ integer associated with each voxel).

### Analyzing data shape
This section of code prints and describes the shape of the two data files. Both data files are 3D arrays, with the first number being the amount of events, the second number being the number of points per event, and the last number being the number of dimensions per point. In this case, we have 6 dimensions per point, and the first 3 dimensions are the x-, y-, and z- coordinates. The 4th dimension is the `voxel_id`.  When run, the block of code below will output print statements that clearly describe what each element in the arrays are. 

In [ ]:
# Combining parameters as strings 
ISOTOPE = 'Mg22'
sample_size = '512'
name = ISOTOPE + '_size' + str(sample_size)
K = 3     # Voxel resolution (the original data will be broken up into K x K x K voxels)

# Using combined parameters to find and load the specific file names
voxelated_data = np.load('voxel_data/' + name + '_voxelated.npy')
basevoxels_data = np.load('voxel_data/' + name + '_base_voxels.npy')

# Slicing the entire array just so we have the x-, y-, and z-coordinates (used for plotting)
voxelated_data_dim = voxelated_data[:,:,:3]      # The first 3 dimensions associated with each point correspond to the x-, y-, z-coordinates
basevoxels_data_dim = basevoxels_data[:,:,:3]
print("voxelated_data")
print("- number of events: " + str(voxelated_data_dim.shape[0]))
print("- points per event: " + str(voxelated_data_dim.shape[1]))
print("- dimensions per point: " + str(voxelated_data_dim.shape[2]))
print("-------------------------")
print("basevoxels_data")
print("- number of events: " + str(basevoxels_data_dim.shape[0]))
print("- points per event: " + str(basevoxels_data_dim.shape[1]))
print("- dimensions per point: " + str(basevoxels_data_dim.shape[2]))

# Slicing the entire array to isolate the voxel_id dimension (will be used later)
voxelated_voxel_id = voxelated_data[:,:,3]       # The 4th dimension is the voxel_id, as assigned by the Mg22_Voxel_pipeline.ipynb notebook
basevoxels_voxel_id = basevoxels_data[:,:,3]

### 3D Scatterplot
We can use _matplotlib_ to create a 3D scatterplot of any given event by taking the first 3 dimensions. In the block below, we can change the value of `ev_num` and run the code to see how the plots compare between the `voxelated_data` and the `basevoxels_data` for any given event. The top plot is the _voxelated_data_ file and the bottom plot is the _base_voxels_ file.  

In [ ]:
# Choosing the nth event and splitting the position into three arrays of x, y, z positions
ev_num = 1994     # the event number can range from 0 to 2426
voxelated_x = voxelated_data_dim[ev_num,:,0]
voxelated_y = voxelated_data_dim[ev_num,:,1]
voxelated_z = voxelated_data_dim[ev_num,:,2]

basevoxels_x = basevoxels_data_dim[ev_num,:,0]
basevoxels_y = basevoxels_data_dim[ev_num,:,1]
basevoxels_z = basevoxels_data_dim[ev_num,:,2]

# 3D scatter plot of both data sets
fig_voxelated = plt.axes(projection='3d')
fig_voxelated.set_xlabel('x')
fig_voxelated.set_ylabel('y')
fig_voxelated.set_zlabel('z')
fig_voxelated.scatter3D(voxelated_x, voxelated_y, voxelated_z, s=5)      # s changes the size of each point
plt.show()

fig_basevoxels = plt.axes(projection='3d')
fig_basevoxels.set_xlabel('x')
fig_basevoxels.set_ylabel('y')
fig_basevoxels.set_zlabel('z')
fig_basevoxels.scatter3D(basevoxels_x, basevoxels_y, basevoxels_z, s=5)
plt.show()

If you look at the axis labels on both plots, you can see that while the top plot's axis values change depending on the event, the bottom plot's axis values are always between 0 and 1/3 in all three directions. Essentially, the top plot is the original, unchanged data (unchanged other than the fact that the data points were normalized to fit in a unit cube). However, the bottom plot is all of the voxels plotted together, all in the first voxel position. In the `Mg22_Voxel_pipeline.ipynb` notebook, the original data is broken up into K x K x K voxels. After the data is broken into voxels, these voxels are shuffled around. In other words, the voxels will randomly swap places with each other (remember, our end goal is to have an ML algorithm that would take a shuffled event and put them back together to match the original data). One step that we take in order to make the shuffling process easier is to have a data file in which we have all of the voxels plotted on top of each other. So, while the actual plot that we observe here doesn't actually tell us anything useful about the data, it is a necessary step to make the shuffling process easier.